In [1]:
# ---------------------------------------------------------
# Step 1: Create a sample text file for the assignment
# ---------------------------------------------------------
file_content = "unhappily walked quickly cats"
with open('morph_input.txt', 'w') as f:
    f.write(file_content)

print(f"Created input file with text: '{file_content}'\n")

# ---------------------------------------------------------
# Step 2: Define the Morphological Grammar (in Strict CNF)
# ---------------------------------------------------------
# Chomsky Normal Form (CNF) rules: A -> BC or A -> a
# We explicitly handle Derivational (changes meaning/class) 
# and Inflectional (changes tense/plurality) morphemes.

morph_grammar = {
    # 1. Terminal Rules (A -> a)
    'PREFIX_DERIV': [['un-']],     # Derivational: changes meaning to opposite
    'SUFFIX_DERIV': [['-ly']],     # Derivational: changes adjective to adverb
    'SUFFIX_INFL_PAST': [['-ed']], # Inflectional: changes tense
    'SUFFIX_INFL_PL': [['-s']],    # Inflectional: changes plurality
    
    'ROOT_ADJ': [['happy'], ['quick']],
    'ROOT_VERB': [['walk']],
    'ROOT_NOUN': [['cat']],

    # 2. Non-Terminal Binary Rules (A -> BC)
    # Handling prefixed adjectives (un- + happy = unhappy)
    'ADJ_STEM': [['PREFIX_DERIV', 'ROOT_ADJ']], 
    
    # Handling derivations (unhappy + -ly = unhappily) OR (quick + -ly = quickly)
    'MORPH_WORD': [
        ['ADJ_STEM', 'SUFFIX_DERIV'], 
        ['ROOT_ADJ', 'SUFFIX_DERIV'], 
        
        # Handling inflections (walk + -ed = walked) OR (cat + -s = cats)
        ['ROOT_VERB', 'SUFFIX_INFL_PAST'],
        ['ROOT_NOUN', 'SUFFIX_INFL_PL']
    ]
}

# ---------------------------------------------------------
# Step 3: Define a simple segmenter
# ---------------------------------------------------------
def segment_word(word):
    """
    A simple dictionary-based segmenter to break words into morphemes.
    Keeps the assignment straightforward without needing complex ML models.
    """
    segmentation_rules = {
        "unhappily": ["un-", "happy", "-ly"], 
        "walked": ["walk", "-ed"],            
        "quickly": ["quick", "-ly"],          
        "cats": ["cat", "-s"]                 
    }
    return segmentation_rules.get(word, [word])

# ---------------------------------------------------------
# Step 4: The CKY Parsing Algorithm
# ---------------------------------------------------------
def cky_morph_parse(morphemes, grammar):
    n = len(morphemes)
    if n == 0:
        return False
        
    # Initialize the CKY table
    table = [[set() for _ in range(n)] for _ in range(n)]

    # Fill the diagonal (Length 1 spans - matching terminals)
    for i, morpheme in enumerate(morphemes):
        for non_terminal, productions in grammar.items():
            for production in productions:
                if len(production) == 1 and production[0] == morpheme:
                    table[0][i].add(non_terminal)

    # Fill the upper triangle (Lengths 2 to n - matching non-terminals)
    for length in range(2, n + 1):
        for i in range(n - length + 1):
            j = i + length - 1
            for k in range(i, j):
                # Look at left and right sub-spans
                left_constituents = table[k - i][i]
                right_constituents = table[j - k - 1][k + 1]

                # Check if any A -> BC rule matches our left and right spans
                for non_terminal, productions in grammar.items():
                    for production in productions:
                        if len(production) == 2:
                            B, C = production[0], production[1]
                            if B in left_constituents and C in right_constituents:
                                table[length - 1][i].add(non_terminal)

    # Print the CKY table to show work
    print(f"CKY Table for: {' + '.join(morphemes)}")
    for length_idx in range(n):
        print(f"  Span Length {length_idx + 1}:")
        for start_idx in range(n - length_idx):
            end_idx = start_idx + length_idx
            span_morphemes = ''.join(morphemes[start_idx : end_idx + 1])
            print(f"    Morpheme(s) [{span_morphemes}]: {table[length_idx][start_idx]}")

    # Check if a complete word ('MORPH_WORD') was formed across the whole span
    return 'MORPH_WORD' in table[n - 1][0]

# ---------------------------------------------------------
# Step 5: Execute the pipeline
# ---------------------------------------------------------
with open('morph_input.txt', 'r') as f:
    text = f.read().strip()

print("--- Starting Morphological CKY Analysis ---\n")

for word in text.split():
    print(f"Analyzing word: '{word}'")
    morphemes = segment_word(word)
    
    is_valid = cky_morph_parse(morphemes, morph_grammar)
    
    print(f"Result: Morphologically valid = {is_valid}")
    print("-" * 40 + "\n")

Created input file with text: 'unhappily walked quickly cats'

--- Starting Morphological CKY Analysis ---

Analyzing word: 'unhappily'
CKY Table for: un- + happy + -ly
  Span Length 1:
    Morpheme(s) [un-]: {'PREFIX_DERIV'}
    Morpheme(s) [happy]: {'ROOT_ADJ'}
    Morpheme(s) [-ly]: {'SUFFIX_DERIV'}
  Span Length 2:
    Morpheme(s) [un-happy]: {'ADJ_STEM'}
    Morpheme(s) [happy-ly]: {'MORPH_WORD'}
  Span Length 3:
    Morpheme(s) [un-happy-ly]: {'MORPH_WORD'}
Result: Morphologically valid = True
----------------------------------------

Analyzing word: 'walked'
CKY Table for: walk + -ed
  Span Length 1:
    Morpheme(s) [walk]: {'ROOT_VERB'}
    Morpheme(s) [-ed]: {'SUFFIX_INFL_PAST'}
  Span Length 2:
    Morpheme(s) [walk-ed]: {'MORPH_WORD'}
Result: Morphologically valid = True
----------------------------------------

Analyzing word: 'quickly'
CKY Table for: quick + -ly
  Span Length 1:
    Morpheme(s) [quick]: {'ROOT_ADJ'}
    Morpheme(s) [-ly]: {'SUFFIX_DERIV'}
  Span Length 2:
